# GPS Tracks — All Raw Folders

Processes each of the 9 subfolders in `Datasets/raw_files/` independently.
For each folder, GPS coordinates are extracted from every `.raw` file, the raw
bytes are freed immediately after parsing to keep memory usage manageable, and
a separate map is produced showing the vessel track for that batch.

In [ ]:
import sys
import gc
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import contextily as cx

# Add project root to path
sys.path.insert(0, str(Path('../../').resolve()))
from project_paths import REPO_ROOT

from raw_sensor_integration.raw_parser import read_raw_file
from raw_sensor_integration.gps_parser import parse_nme0_datagrams, parse_gps_sentences

RAW_FILES_DIR = REPO_ROOT / 'Datasets/raw_files'
folders = sorted(f for f in RAW_FILES_DIR.iterdir() if f.is_dir())
print(f"Found {len(folders)} folders:")
for f in folders:
    raw_count = len(list(f.glob('*.raw')))
    print(f"  {f.name}  ({raw_count} .raw files)")

In [ ]:
# ---------------------------------------------------------------------------
# GPS extraction — memory-safe: raw bytes freed immediately after parsing
# ---------------------------------------------------------------------------
folder_tracks = {}  # folder_name -> DataFrame

for folder in folders:
    raw_files = sorted(folder.glob('*.raw'))
    if not raw_files:
        continue

    print(f"\n{folder.name}  ({len(raw_files)} files)")
    dfs = []

    for raw_file in raw_files:
        print(f"  {raw_file.name}...", end=" ", flush=True)
        data = read_raw_file(raw_file)       # ~100 MB bytes
        records = parse_nme0_datagrams(data)
        del data                             # free raw bytes immediately
        gc.collect()

        df = parse_gps_sentences(records)
        del records
        gc.collect()

        if len(df) > 0:
            df['file'] = raw_file.name
            dfs.append(df)
            print(f"{len(df)} fixes")
        else:
            print("no GPS")

    if dfs:
        track = pd.concat(dfs, ignore_index=True).sort_values('timestamp').reset_index(drop=True)
        folder_tracks[folder.name] = track
        print(f"  → {len(track)} total fixes")
    else:
        print("  → No GPS data found")

print(f"\nDone. {len(folder_tracks)}/{len(folders)} folders have GPS data.")

In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

colors = plt.cm.tab10.colors
_folder_list = list(folder_tracks.items())

def plot_track(folder_name, track):
    fig, ax = plt.subplots(figsize=(12, 9), dpi=130)
    for i, (fname, group) in enumerate(track.groupby('file')):
        g = group.sort_values('timestamp')
        c = colors[i % len(colors)]
        ax.plot(g['lon'], g['lat'], '-', color=c, linewidth=1.5, alpha=0.85,
                label=fname, zorder=3)
        ax.plot(g['lon'].iloc[0],  g['lat'].iloc[0],  '^', color=c, markersize=8, zorder=5)
        ax.plot(g['lon'].iloc[-1], g['lat'].iloc[-1], 's', color=c, markersize=8, zorder=5)

    # Guard against degenerate (zero-area) bounding box — contextily hangs indefinitely
    # when all coordinates are identical (e.g. GPS stuck on last fix)
    lon_range = track['lon'].max() - track['lon'].min()
    lat_range = track['lat'].max() - track['lat'].min()
    if lon_range < 1e-6 or lat_range < 1e-6:
        pad = 0.01  # ~1 km padding
        ax.set_xlim(track['lon'].mean() - pad, track['lon'].mean() + pad)
        ax.set_ylim(track['lat'].mean() - pad, track['lat'].mean() + pad)
        print(f"WARNING: {folder_name} has degenerate GPS (all fixes identical) — basemap skipped.")
    else:
        cx.add_basemap(ax, crs='EPSG:4326',
                       source=cx.providers.CartoDB.Positron,
                       zoom=13, attribution_size=6)

    short = folder_name.replace('drive-download-', '').split('Z')[0]
    ax.set_title(f'GPS Track — batch {short}  (▲ = start, ■ = end)', fontsize=12)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    ax.legend(fontsize=6, loc='upper left', framealpha=0.9, title='File')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    gc.collect()

print(f"{len(_folder_list)} folders ready to plot — run each cell below independently.")

In [ ]:
# Folder 1 / 9
plot_track(*_folder_list[0])

In [ ]:
# Folder 2 / 9
plot_track(*_folder_list[1])

In [ ]:
# Folder 3 / 9
plot_track(*_folder_list[2])

In [ ]:
# Folder 4 / 9
plot_track(*_folder_list[3])

In [ ]:
# Folder 5 / 9
plot_track(*_folder_list[4])

In [ ]:
# Folder 6 / 9
plot_track(*_folder_list[5])

In [ ]:
# Folder 7 / 9
plot_track(*_folder_list[6])

In [ ]:
# Folder 8 / 9
plot_track(*_folder_list[7])

In [ ]:
# Folder 9 / 9
plot_track(*_folder_list[8])

In [ ]:
# ---------------------------------------------------------------------------
# (Optional) Save GPS tracks to CSV — one file per folder
# ---------------------------------------------------------------------------
out_dir = Path('.')
for folder_name, track in folder_tracks.items():
    short = folder_name.replace('drive-download-', '').split('Z')[0]
    out = out_dir / f'gps_{short}.csv'
    track.to_csv(out, index=False)
    print(f"Saved {len(track)} rows → {out.resolve()}")